# LightGBM LambdaRank · diario de experimentos (Adri · B)

**Qué es:** un `LGBMRanker` que puntúa **cada tarea del snapshot** a partir de sus columnas (y de features que la comparan
con el resto de tareas del mismo snapshot). Dentro de cada snapshot se ordenan las tareas por puntuación y las 3 primeras
son la predicción. Es el modelo de referencia del equipo y el que mejor funciona.

**Cómo usar este notebook**
1. Ejecuta *Preparación* una vez (carga datos y define features y funciones, ~30 s).
2. Cada bloque tiene: **hipótesis** → **variantes** (se cambia UNA cosa respecto a la base del bloque) → **tabla** → **conclusión**.
3. Los bloques 1 y 2 ya están hechos (sus conclusiones están escritas); no se repiten salvo que pongas `REPETIR_BLOQUES_1_2 = True`.
4. Lo que gane se apunta en `MEJOR` (bloque 5) y el bloque 6 genera la versión para el stacking.

**Reglas que respeta el modelo** (`CLAUDE.md` / `EQUIPO.md`)
- Cada snapshot se predice **solo con sus propias filas**: toda feature nueva se calcula con `groupby("sample_id")`
  (o con grupos más finos dentro del snapshot: avión, tipo).
- CV con `Data/folds.csv` (5 folds por controlador; cada sesión queda entera en un fold).
- No usa `sample_id, session, atco, scenario, timestamp, task_name, row_index` como features
  (`task_name` solo se usa para sacar el avión y agrupar).

**Cómo leer las cifras**
- `cv` = top-3 hit rate out-of-fold sobre train+validation (13 175 snapshots).
- `peor_atco` = nota del peor controlador. Importa: el leaderboard privado es **un solo controlador** (023).
- Con 1 semilla, diferencias < ~0.005 son ruido; con 2 semillas, < ~0.003. Veredicto: **mejora** si Δ ≥ `MIN_GAIN`.
- Cada experimento se apunta en `experiments/lgb_runs.csv` (celda *Historial* al final).

> **Resumen (20/10, 19:15):** versión en `main` = **`lgb_adri` · base + `alerta` + `contexto` · CV 0.657** (2 semillas),
> validation.csv 0.677. Con 2 semillas **empata con la base** (0.657): las features añadidas mueven poco la nota total,
> aunque mejoran entry_coordination y conflictos. Lo que no funciona: meter todos los grupos a la vez, `exit_coord`, `deadline`.
> **Análisis de errores (bloque 3):** el 83 % de los fallos es por elegir **el avión equivocado**, no la tarea dentro del avión;
> exit_coordination (9 % de acierto) y entry_coordination (39 %) son los que más fallos acumulan.

## Preparación

### Configuración
`PARAMS` son los parámetros base de `EQUIPO.md`. Cada experimento cambia **una** clave respecto a ellos.

In [ ]:
import json, subprocess, sys, time
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
for p in ("oof", "preds", "submissions", "models", "experiments"):
    (ROOT / p).mkdir(exist_ok=True)

NAME = "lgb_adri"   # nombre de los archivos de salida
TYPES = ['entry_conflict_resolution', 'exit_conflict_resolution', 'entry_coordination', 'exit_coordination',
         'assume_aircraft', 'transfer_aircraft', 'conflict_resolution', 'non_conformance_resolution',
         'qos_improvement', 'entry_conditions', 'exit_conditions', 'aircraft_request', 'zone_conflict',
         'return_to_route']

PARAMS = dict(objective="lambdarank", n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, lambdarank_truncation_level=5,
              random_state=0, verbose=-1)
SEEDS_BLOQUE = [0, 1]        # semillas de cada variante en los bloques nuevos (se promedian las predicciones)
SEEDS_FINAL = [0, 1, 2]      # semillas de la versión para el stacking
MIN_GAIN = 0.003             # Δ mínimo para decir "mejora" con 2 semillas
REPETIR_BLOQUES_1_2 = False  # True = volver a lanzar los bloques 1 y 2 (~10 min)

### Datos
- `lab` = train + validation (con etiqueta), cada fila con su `fold` (0–4) de `Data/folds.csv`.
- `test` = test, sin etiqueta.
- `cs` = callsign del avión de la tarea (sacado de `task_name`). **No es una feature**: solo sirve para agrupar las tareas
  de un mismo avión.

In [ ]:
def load(name):
    d = pd.read_csv(DATA / f"{name}.csv")
    pref = d.task_index.map(lambda t: len(TYPES[t]) + 1)
    d["cs"] = [n[p:].split("_")[0] for n, p in zip(d.task_name, pref)]
    return d

lab = pd.concat([load("train"), load("validation")], ignore_index=True)
lab = lab.merge(pd.read_csv(DATA / "folds.csv"), on="atco", how="left")
test = load("test")
lab = lab.sort_values(["sample_id", "row_index"]).reset_index(drop=True)
test = test.sort_values(["sample_id", "row_index"]).reset_index(drop=True)
ATCO = lab.drop_duplicates("sample_id").set_index("sample_id").atco

assert lab.fold.notna().all(), "hay controladores sin fold"
assert (lab.groupby("sample_id").target_index.sum() == 1).all(), "alguna muestra no tiene exactamente 1 respuesta"
print(f"lab: {lab.sample_id.nunique():,} muestras, {len(lab):,} filas | test: {test.sample_id.nunique():,} muestras")

### Features
Cada grupo es una función que recibe el dataframe y devuelve **solo las columnas nuevas**. Todas agrupan por `sample_id`
(o por avión / tipo dentro del snapshot), así que solo usan el propio snapshot.

| Grupo | Origen | Qué añade |
|---|---|---|
| `snapshot` | EQUIPO.md | Ranking y diferencia respecto al mínimo **dentro del snapshot** de columnas de urgencia/actividad |
| `tipo` | EQUIPO.md | Los mismos rankings **dentro de las tareas del mismo tipo** |
| `avion` | EQUIPO.md | Si la tarea es la más antigua de su avión, cuántos tipos distintos tiene el avión |
| `tipo_estado` | EQUIPO.md + Pau #2 | Cruces tipo × estado: assume con oferta, transfer debido, entry_coordination con gemelo (`ts_twin_entry` = `is_entry_losing` de Pau)… |
| `contexto` | EQUIPO.md | Cómo está el snapshot: nº de aviones, transfers/assumes/zonas/conflictos pendientes |
| `exit_coord` | Pau #3 | exit_coordination × coordinación con el sector siguiente (`k_out_*`, mismatch, herramienta), gemelo con transfer, edad frente a la tarea más nueva del avión |
| `alerta` | Pau #4 | Conflictos con alerta STCA en pantalla, nº de alertas del snapshot, ranking del tiempo al mínimo entre las tareas de conflicto |
| `deadline` | Adri | Un "tiempo límite" común a todos los tipos y su ranking en el snapshot y en el tipo |
| `avion_tipos` | bloque 4 | **Qué otras tareas tiene abiertas el mismo avión** (assume, transfer, exit_coord…), nº de tareas del avión y la más antigua |

Pau #5 (ranking de distancia al borde dentro del tipo) ya es una columna original (`l_rank_absf1_type`).
Pau #1 (oráculo) se descarta: usa `scenario` y `timestamp`, prohibidos como features.

In [ ]:
META = ['sample_id', 'target_index', 'session', 'atco', 'scenario', 'timestamp', 'task_name', 'row_index',
        'fold', 'cs']
BASE = [c for c in lab.columns if c not in META]
RANK_COLS = ["a_since_flight", "x_task_age", "x_state_age", "time_to_MD", "k_timeToExitPoint",
             "distance_to_zone", "abs_tts"]
ENTRY_TYPES, EXIT_TYPES, CONFLICT_TYPES = [0, 2, 4, 9], [1, 3, 5, 10], [0, 1, 6]


def _with_abs(df):
    return df.assign(abs_tts=df.time_to_sector.abs())


def _ac_any(df, mask):
    """1 si alguna tarea del mismo avión (en el mismo snapshot) cumple mask."""
    return mask.astype(int).groupby([df.sample_id, df.cs]).transform("max")


def fe_snapshot(df):
    x = _with_abs(df)
    g = x.groupby("sample_id")
    out = {}
    for c in RANK_COLS:
        out[f"rs_{c}"] = g[c].rank(method="min")
        out[f"ds_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_tipo(df):
    x = _with_abs(df)
    g = x.groupby(["sample_id", "task_index"])
    out = {}
    for c in RANK_COLS:
        out[f"rt_{c}"] = g[c].rank(method="min")
        out[f"dt_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_avion(df):
    g = df.groupby(["sample_id", "cs"])
    out = {
        "ac_age_rank": g.x_task_age.rank(method="min", ascending=False),   # 1 = la tarea más antigua del avión
        "ac_n_types": g.task_index.transform("nunique"),
        "ac_has_assume": _ac_any(df, df.task_index == 4),
    }
    return pd.DataFrame(out, index=df.index)


def fe_tipo_estado(df):
    t = df.task_index
    out = {
        "ts_assume_offered": ((t == 4) & (df.x_transfer_in == 1)).astype(int),
        "ts_transfer_due": ((t == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "ts_entry_coord_done": ((t == 2) & (df.x_isc_coordinated == 1)).astype(int),
        "ts_entry_notified": ((t == 2) & (df.x_isc_notified == 1)).astype(int),
        "ts_zone_dist": df.distance_to_zone.where(t == 12),
        "ts_conflict_ttmd": df.time_to_MD.where(t.isin(CONFLICT_TYPES)),
        "ts_twin_assume": ((t == 4) & (df.l_twin_AE == 1)).astype(int),
        "ts_twin_entry": ((t == 2) & (df.l_twin_AE == 1)).astype(int),     # = is_entry_losing de Pau
    }
    return pd.DataFrame(out, index=df.index)


def fe_contexto(df):
    g = df.groupby("sample_id")
    tmp = pd.DataFrame({
        "sample_id": df.sample_id,
        "is_assume_offer": ((df.task_index == 4) & (df.x_transfer_in == 1)).astype(int),
        "is_transfer_due": ((df.task_index == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "is_zone": (df.task_index == 12).astype(int),
        "is_conflict": df.task_index.isin(CONFLICT_TYPES).astype(int),
        "is_entry_coord": (df.task_index == 2).astype(int),
    })
    gt = tmp.groupby("sample_id")
    out = {"cx_n_aircraft": g.cs.transform("nunique")}
    for c in tmp.columns[1:]:
        out["cx_n_" + c[3:]] = gt[c].transform("sum")
    return pd.DataFrame(out, index=df.index)


def fe_exit_coord(df):
    t = df.task_index
    ec = t == 3
    out = {
        "ec_notified": (ec & (df.k_out_NOTIFIED == 1)).astype(int),
        "ec_coordinated": (ec & (df.k_out_COORDINATED == 1)).astype(int),
        "ec_mismatch": (ec & (df.k_outCoordMismatch == 1)).astype(int),
        "ec_tool": (ec & (df.k_coordinationToolActivation == 1)).astype(int),
        "ec_revision": df.k_outCoordRevision.where(ec),
        "ec_with_transfer": (ec & (_ac_any(df, t == 5) == 1)).astype(int),
        "tr_with_exit_coord": ((t == 5) & (_ac_any(df, ec) == 1)).astype(int),
        "age_vs_newest_ac": df.x_task_age - df.groupby(["sample_id", "cs"]).x_task_age.transform("min"),
    }
    return pd.DataFrame(out, index=df.index)


def fe_alerta(df):
    conf = df.task_index.isin(CONFLICT_TYPES)
    stca = df.HMI_conflict_safety_alert == 1
    ttmd = df.time_to_MD.where(conf)
    out = {
        "al_conflict_stca": (conf & stca).astype(int),
        "al_n_stca": stca.astype(int).groupby(df.sample_id).transform("sum"),
        "al_ttmd_rank": ttmd.groupby(df.sample_id).rank(method="min"),
        "al_conflict_multi": (conf & (df.multiple_conflicts_ind == 1)).astype(int),
        "al_conflict_dmd": df.distance_to_MD.where(conf),
    }
    return pd.DataFrame(out, index=df.index)


def fe_deadline(df):
    t = df.task_index
    dl = pd.Series(np.nan, index=df.index)
    entry, exit_, zone = t.isin(ENTRY_TYPES), t.isin(EXIT_TYPES), t == 12
    dl[entry] = df.time_to_sector[entry]
    dl[exit_] = df.k_timeToExitPoint.fillna(df.time_to_sector.abs())[exit_]
    dl[t == 6] = df.time_to_MD[t == 6]
    dl[zone] = (df.distance_to_zone * 1852 / df.k_groundSpeed.fillna(230))[zone]   # NM -> m, m/s
    out = {
        "dl": dl,
        "dl_rank": dl.groupby(df.sample_id).rank(method="min"),
        "dl_rank_type": dl.groupby([df.sample_id, t]).rank(method="min"),
        "dl_minus_min": dl - dl.groupby(df.sample_id).transform("min"),
    }
    return pd.DataFrame(out, index=df.index)


def fe_avion_tipos(df):
    g = df.groupby(["sample_id", "cs"])
    out = {}
    for t, nm in [(2, "entry_coord"), (3, "exit_coord"), (4, "assume"), (5, "transfer"), (10, "exit_cond"),
                  (12, "zone"), (13, "return_route"), (6, "conflict")]:
        out[f"av_has_{nm}"] = _ac_any(df, df.task_index == t)
    out["av_n_tasks"] = g.task_index.transform("size")
    out["av_max_age"] = g.x_task_age.transform("max")
    return pd.DataFrame(out, index=df.index)


FEATURE_GROUPS = {"snapshot": fe_snapshot, "tipo": fe_tipo, "avion": fe_avion,
                  "tipo_estado": fe_tipo_estado, "contexto": fe_contexto, "exit_coord": fe_exit_coord,
                  "alerta": fe_alerta, "deadline": fe_deadline, "avion_tipos": fe_avion_tipos}


def build(df, groups):
    """Devuelve (df con las columnas nuevas, lista de features a usar)."""
    new = [FEATURE_GROUPS[g](df) for g in groups]
    out = pd.concat([df] + new, axis=1)
    return out, BASE + [c for n in new for c in n.columns]

### Entrenamiento, CV y experimentos
- `fit`: entrena un `LGBMRanker`. `group` le dice qué filas forman cada snapshot, para que aprenda a **ordenar dentro del snapshot**.
- `hit3`: para cada snapshot, 1 si la respuesta está entre las 3 de mayor score.
- `cv`: para cada fold, entrena con los otros 4 y predice ese; con varias semillas promedia sus predicciones. El resultado es el **OOF**.
- `experimento`: una variante → CV, peor controlador, y la apunta en `experiments/lgb_runs.csv`.
- `lanzar`: un bloque entero → base del bloque + variantes → **tabla con Δ y veredicto**.

In [ ]:
def fit(A, F, params):
    m = lgb.LGBMRanker(**params)
    m.fit(A[F], A.target_index, group=A.groupby("sample_id", sort=False).size().values,
          categorical_feature=["task_index"])
    return m


def hit3(d, score):
    t = d[["sample_id", "target_index"]].assign(s=np.asarray(score))
    t = t.sort_values(["sample_id", "s"], ascending=[True, False])
    return t.groupby("sample_id").head(3).groupby("sample_id").target_index.max()


def cv(d, F, params, seeds=(0,)):
    oof = np.zeros(len(d))
    for f in range(5):
        tr, va = (d.fold != f).values, (d.fold == f).values
        for s in seeds:
            oof[va] += fit(d[tr], F, {**params, "random_state": s}).predict(d.loc[va, F]) / len(seeds)
    h = hit3(d, oof)
    fold_of = d.drop_duplicates("sample_id").set_index("sample_id").fold
    return oof, h, h.groupby(fold_of.reindex(h.index)).mean()


LOG = ROOT / "experiments" / "lgb_runs.csv"
OOFS = {}   # nombre -> (d, oof, aciertos por muestra), para analizar después


def experimento(nombre, groups, params=None, seeds=SEEDS_BLOQUE, nota=""):
    params = {**PARAMS, **(params or {})}
    t0 = time.time()
    d, F = build(lab, groups)
    oof, h, por_fold = cv(d, F, params, seeds)
    OOFS[nombre] = (d, oof, h)
    peor = h.groupby(ATCO.reindex(h.index)).mean().min()
    fila = dict(hora=time.strftime("%H:%M"), nombre=nombre, grupos="+".join(groups) or "-",
                params_cambiados=str({k: v for k, v in params.items() if PARAMS.get(k) != v}),
                n_features=len(F), cv_top3=round(h.mean(), 4), std_folds=round(por_fold.std(), 4),
                nota=f"{len(seeds)} semillas. peor_atco {peor:.3f}. {nota}".strip())
    pd.DataFrame([fila]).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)
    print(f"{nombre:40s} CV {h.mean():.4f}  peor_atco {peor:.3f}  (folds {' '.join(f'{v:.3f}' for v in por_fold)})"
          f"  {len(F)} feats  {time.time() - t0:.0f}s", flush=True)
    return h.mean(), peor


def lanzar(bloque, variantes, grupos, params=None, seeds=SEEDS_BLOQUE):
    """variantes = [(nombre, por qué, grupos extra, params extra), ...]. Devuelve la tabla del bloque."""
    print(f"=== bloque {bloque}: base = {grupos or 'columnas originales'} {params or ''}  ({len(seeds)} semillas)")
    filas = [("base", *experimento(f"{bloque}/base", grupos, params, seeds, "referencia del bloque"))]
    for nombre, por_que, g_extra, p_extra in variantes:
        filas.append((nombre, *experimento(f"{bloque}/{nombre}", grupos + g_extra, {**(params or {}), **p_extra},
                                           seeds, por_que)))
    t = pd.DataFrame(filas, columns=["variante", "cv", "peor_atco"])
    t["Δ_vs_base"] = t.cv - t.cv.iloc[0]
    t["veredicto"] = ["" if i == 0 else "mejora" if v >= MIN_GAIN else "empeora" if v <= -MIN_GAIN else "ruido"
                      for i, v in enumerate(t["Δ_vs_base"])]
    return t.round(4)


def historial(n=20):
    return pd.read_csv(LOG).tail(n)

## 0 · Punto de partida

LightGBM LambdaRank con **las columnas originales** y los parámetros de `EQUIPO.md`, folds de `Data/folds.csv`:
**CV 0.651** con 1 semilla, **0.657** con 2 semillas (folds 0.683 / 0.659 / 0.647 / 0.640 / 0.630).

**Diagnóstico:** el modelo acierta muy bien los tipos frecuentes (transfer 86 %, assume 83 %, zone 71 %) y casi nunca
los raros (exit_coordination 10 %, exit_conditions 19 %). La nota varía mucho entre controladores (0.63–0.70 según el fold):
el controlador del privado puede caer en cualquier punto de ese rango.

## 1 · Features de `EQUIPO.md` (ronda 1)

**Hipótesis:** el árbol puntúa cada tarea por separado; darle hechas las comparaciones con el resto del snapshot
(rankings, diferencias, contexto) debería ayudarle a ordenar.

| Cambio | Por qué podría ayudar |
|---|---|
| `snapshot` | posición relativa de la tarea en el snapshot (urgencia, antigüedad, actividad reciente) |
| `tipo` | lo mismo pero solo frente a las tareas del mismo tipo (FIFO por tipo) |
| `avion` | distinguir las tareas de un mismo avión |
| `tipo_estado` | señales que solo tienen sentido para un tipo (assume con oferta pendiente, transfer debido…) |
| `contexto` | cómo de cargado está el snapshot |

1 semilla, ~5 min.

In [ ]:
if REPETIR_BLOQUES_1_2:
    tabla1 = lanzar("b1", [(g, "ronda 1", [g], {}) for g in ["snapshot", "tipo", "avion", "tipo_estado", "contexto"]]
                    + [("todos", "todos juntos", ["snapshot", "tipo", "avion", "tipo_estado", "contexto"], {})],
                    [], seeds=[0])
    display(tabla1)
else:
    print("Bloque ya ejecutado: ver la conclusión. REPETIR_BLOQUES_1_2 = True para repetirlo.")

**Conclusión bloque 1 · mejoras pequeñas por separado, y juntas empeoran.**

| Variante | CV | Δ vs base (0.651) | |
|---|---|---|---|
| contexto | 0.655 | +0.004 | ruido / mejora justa |
| tipo_estado | 0.655 | +0.004 | ruido / mejora justa |
| tipo | 0.654 | +0.003 | ruido |
| avion | 0.653 | +0.001 | ruido |
| snapshot | 0.652 | +0.000 | ruido |
| **todos juntos** | 0.648 | **−0.004** | **empeora** |

- Con 133 columnas, muchas redundantes, el árbol se sobreajusta a los controladores de entrenamiento.
- `tipo_estado` repetido con 2 semillas da 0.6565 frente a 0.6570 de la base: **no aporta**.
- Las columnas originales `l_rank_*` ya dan buena parte de la posición relativa: por eso `snapshot` y `tipo` aportan poco.

## 2 · Ideas de Pau y `deadline` (ronda 2)

**Hipótesis:** el EDA de Pau (sección 9 de `01_eda.ipynb`) señala dónde falla el modelo; features pensadas para esos
tipos deberían subir su acierto.

| Cambio | Por qué podría ayudar |
|---|---|
| `exit_coord` (Pau #3) | exit_coordination acierta un 10 %: decirle cuándo la coordinación con el sector siguiente está pendiente o desajustada |
| `alerta` (Pau #4) | conflict_resolution acierta un 30 %: la alerta STCA en pantalla es lo que el controlador ve |
| `deadline` (Adri) | un tiempo límite comparable entre tipos distintos (entrada, salida, conflicto, zona) |
| `contexto` | el mejor de la ronda 1, para combinar |

1 semilla, selección hacia delante (cada grupo solo se queda si sube la CV), ~5 min.

In [ ]:
if REPETIR_BLOQUES_1_2:
    tabla2 = lanzar("b2", [(g, "ronda 2", [g], {}) for g in ["contexto", "exit_coord", "alerta", "deadline"]]
                    + [("alerta+contexto", "combinación de los dos mejores", ["alerta", "contexto"], {})],
                    [], seeds=[0])
    display(tabla2)
else:
    print("Bloque ya ejecutado: ver la conclusión. REPETIR_BLOQUES_1_2 = True para repetirlo.")

**Conclusión bloque 2 · `alerta` es la única idea de Pau que funciona; con 2 semillas la mejora se diluye.**

| Variante | CV (1 semilla) | Δ vs base (0.651) | |
|---|---|---|---|
| **alerta + contexto** | **0.660** | **+0.008** | **mejora** |
| alerta | 0.657 | +0.006 | mejora |
| contexto | 0.655 | +0.004 | mejora justa |
| exit_coord | 0.651 | −0.001 | ruido |
| deadline | 0.651 | −0.000 | ruido |

- Confirmado con 2 semillas: **alerta + contexto = 0.6569**, base = 0.6570 → **empate**. Parte de la mejora con 1 semilla
  era suerte (se elige al ganador entre varios intentos ruidosos).
- Aun así cambia *en qué* acierta: entry_coordination 35 % → 39 %, conflict_resolution 32 % → 34 %, return_to_route 51 % → 53 %.
- `exit_coord` no arregla exit_coordination (sigue en ~8–10 %): las columnas `k_out_*` ya estaban en la base; el árbol ya podía usarlas.
- Es la versión que está ahora en `main` (`oof/lgb_adri.csv`).

## 3 · Análisis de errores

**Pregunta:** ¿dónde y cómo falla? Sirve para decidir qué atacar. Se mira sobre el OOF (predicciones hechas sin haber visto
a ese controlador). `analizar_errores` vale para cualquier OOF del equipo (`"nn_tf_v2"`, `"types"`…). ~20 s.

Qué mide:
1. **Avión correcto, tarea equivocada:** si el avión de la respuesta está en el top-3 pero con otra de sus tareas.
2. **Fallos por tipo correcto:** acierto, nº de fallos y qué parte del total de fallos supone cada tipo.
3. **Posición de la respuesta cuando falla:** si se queda cerca (4ª–5ª) o lejos.
4. **Qué pone el modelo en 1ª posición cuando falla.**
5. **Gemelos:** cuando el avión correcto tiene dos tareas que compiten, cuál elige el controlador.

In [ ]:
CORTO = ['entry_conflict', 'exit_conflict', 'entry_coord', 'exit_coord', 'assume', 'transfer', 'conflict', 'non_conf',
         'qos', 'entry_cond', 'exit_cond', 'request', 'zone', 'return_route']


def analizar_errores(nombre):
    d = lab.merge(pd.read_csv(ROOT / "oof" / f"{nombre}.csv"), on=["sample_id", "row_index"])
    d["rk"] = d.groupby("sample_id").score.rank(ascending=False, method="first")
    d["typ"] = d.task_index.map(lambda t: CORTO[t])
    y = d[d.target_index == 1].set_index("sample_id")
    y["hit"] = y.rk <= 3
    ac_top3 = d[d.rk <= 3].groupby("sample_id").cs.apply(set)
    y["avion_ok"] = [cs in ac_top3[s] for s, cs in zip(y.index, y.cs)]
    fallo = ~y.hit
    print(f"{nombre}: top-3 {y.hit.mean():.4f} | {fallo.sum()} fallos, de ellos con el AVIÓN correcto en el top-3: "
          f"{(fallo & y.avion_ok).sum()} ({(fallo & y.avion_ok).sum() / fallo.sum():.0%})")

    g = y.groupby("typ").agg(muestras=("hit", "size"), acierto=("hit", "mean"), fallos=("hit", lambda h: (~h).sum()))
    g["%_de_los_fallos"] = (g.fallos / fallo.sum() * 100).round(1)
    g["avion_ok_tarea_mal"] = (fallo & y.avion_ok).groupby(y.typ).sum()
    g["posición_mediana_si_falla"] = y[fallo].groupby("typ").rk.median()
    display(g.sort_values("fallos", ascending=False).round(3))

    print("Posición de la respuesta cuando falla:")
    display(pd.cut(y[fallo].rk, [3, 4, 5, 6, 10, 20, 99]).value_counts(sort=False).rename("fallos").to_frame().T)

    print("Cuando falla, tipo que el modelo pone 1º (filas = tipo correcto):")
    top1 = d[d.rk == 1].set_index("sample_id").typ
    ct = pd.crosstab(y[fallo].typ, top1.reindex(y[fallo].index))
    display(ct.loc[g.sort_values("fallos", ascending=False).index[:7]])

    print("Gemelos: el avión correcto tiene las dos tareas, ¿cuál elige el controlador?")
    tiene = d.groupby(["sample_id", "cs"]).typ.agg(set)
    filas = []
    for a, b in [("exit_coord", "transfer"), ("entry_coord", "assume"), ("return_route", "transfer"),
                 ("exit_cond", "transfer"), ("exit_coord", "exit_cond"), ("zone", "transfer")]:
        r = y[y.typ.isin([a, b])]
        r = r[[{a, b} <= tiene.get((s, cs), set()) for s, cs in zip(r.index, r.cs)]]
        filas.append(dict(par=f"{a} vs {b}", casos=len(r), elige_el_primero=round((r.typ == a).mean(), 3),
                          acierto_modelo=round(r.hit.mean(), 3)))
    display(pd.DataFrame(filas))


analizar_errores(NAME)

**Conclusión del análisis de errores (`lgb_adri`, CV 0.657):**

| Tipo correcto | Muestras | Acierto top-3 | % de todos los fallos | Posición mediana si falla |
|---|---|---|---|---|
| exit_coordination | 745 | **0.09** | **15 %** | 11ª |
| entry_coordination | 1.068 | 0.39 | **14 %** | 9ª |
| zone_conflict | 2.121 | 0.71 | 14 % | 6ª |
| assume | 3.366 | 0.84 | 12 % | 5ª |
| transfer | 3.527 | 0.86 | 11 % | 5ª |
| return_to_route | 937 | 0.50 | 10 % | 5ª |
| exit_conditions | 448 | 0.19 | 8 % | 10ª |
| conflict_resolution | 390 | 0.32 | 6 % | 6ª |

1. **El problema es el avión, no la tarea:** solo el **17 %** de los fallos tiene el avión correcto en el top-3. En el
   83 % restante el modelo está mirando **otros aviones**. Mejorar el "gemelo" (assume vs entry_coord…) tiene un techo bajo.
2. **Un tercio de los fallos es por poco:** 1.589 fallos (35 %) tienen la respuesta en 4ª–5ª posición.
3. **Juega a lo seguro:** cuando falla, pone en 1ª posición transfer, assume o zone casi siempre (como la red de Pablo).
4. **exit_coordination está enterrada** (mediana 11ª): el modelo casi nunca la sube. Cuando el avión tiene exit_coord y
   transfer, el controlador elige exit_coord solo un **11 %** de las veces: el modelo aprende "transfer gana" y hace bien
   en media, pero pierde esos casos.
5. **Gemelos:** entry_coord vs assume → el controlador elige entry_coord un 27 %; return_to_route vs transfer un 30 %;
   exit_conditions vs transfer un 16 %.

**Qué atacar:** (a) que el modelo sepa qué **otras tareas tiene el avión** (el contexto de cada avión, no de cada tarea);
(b) que la función de pérdida mire **el top-3** y no el top-5; (c) corregir el sesgo hacia los tipos frecuentes.

## 4 · Atacar los fallos

**Hipótesis:** según el bloque 3, el modelo falla sobre todo al elegir el avión y entierra los tipos raros. Se prueban
cambios dirigidos a eso **sobre la versión actual** (`alerta` + `contexto`), con 2 semillas.

| Cambio | Por qué podría ayudar |
|---|---|
| `avion_tipos` | el árbol ve cada tarea aislada; saber **qué otras tareas tiene abiertas el avión** (assume, transfer, exit_coord…) da el contexto del avión. Ataca el 83 % de fallos de avión y los gemelos |
| `truncation_level=3` | LambdaRank optimiza el orden de las 5 primeras; nuestra métrica solo mira las **3 primeras** |
| `truncation_level=10` | al revés: dar peso a respuestas que caen en 6ª–10ª (exit_coord, entry_coord) para subirlas |
| `rank_xendcg` | otra pérdida de ranking, más robusta al **ruido en las etiquetas** (los controladores no coinciden mucho) |
| `regularizado` | `num_leaves` 15, `min_child_samples` 100, `lr` 0.03 × 700 árboles: árboles más simples, menos memorizar controladores |

~1 min por variante, ~6–7 min el bloque.

In [ ]:
GRUPOS_ACTUALES = ["alerta", "contexto"]   # la versión de main (bloque 2)

tabla4 = lanzar("b4", [
    ("avion_tipos",  "qué otras tareas tiene el avión",          ["avion_tipos"], {}),
    ("trunc=3",      "la pérdida mira solo el top-3",            [], dict(lambdarank_truncation_level=3)),
    ("trunc=10",     "dar peso a respuestas en 6ª-10ª",          [], dict(lambdarank_truncation_level=10)),
    ("xendcg",       "pérdida más robusta al ruido",             [], dict(objective="rank_xendcg")),
    ("regularizado", "árboles más simples, menos memorizar",     [],
     dict(num_leaves=15, min_child_samples=100, learning_rate=0.03, n_estimators=700)),
], GRUPOS_ACTUALES)
tabla4

**Conclusión bloque 4 · (rellenar tras ejecutar)**

| Variante | CV | Δ vs base | peor controlador | |
|---|---|---|---|---|
| … | | | | |

-

### 4b · Corregir el sesgo hacia los tipos frecuentes (post-proceso)

**Hipótesis:** el modelo juega a lo seguro (transfer, assume, zone). Sumar un pequeño **sesgo por tipo** a la puntuación
podría subir exit_coordination, exit_conditions… sin reentrenar. Los sesgos se ajustan en 4 folds y se miden en el 5º
(CV anidada), así que la nota es honesta. Se aplica sobre el OOF de la base del bloque 4. ~1 min, no entrena nada.

In [ ]:
def sesgo_por_tipo(d, oof, grid=(-1.0, -0.5, -0.25, 0, 0.25, 0.5, 1.0), tipos=(3, 10, 2, 12, 13, 6, 8, 1, 4, 5)):
    fold, ti = d.fold.values, d.task_index.values

    def acierto(score, mask):
        return hit3(d[mask], score[mask]).mean()

    out = np.zeros(len(d))
    for f in range(5):
        trm = fold != f
        b, mejor_b = np.zeros(14), acierto(oof, trm)
        for _ in range(2):
            for t in tipos:
                for v in grid:
                    bb = b.copy(); bb[t] = v
                    s = acierto(oof + bb[ti], trm)
                    if s > mejor_b + 1e-4:
                        mejor_b, b = s, bb
        out[fold == f] = (oof + b[ti])[fold == f]
        print(f"fold {f}: sesgos " + ", ".join(f"{TYPES[t]} {v:+}" for t, v in enumerate(b) if v))
    return hit3(d, out).mean()


d4, oof4, h4 = OOFS["b4/base"]
print(f"\nsin sesgo: {h4.mean():.4f}   con sesgo por tipo (CV anidada): {sesgo_por_tipo(d4, oof4):.4f}")

**Conclusión bloque 4b · (rellenar tras ejecutar)**

-

## 5 · Combinar lo que ha funcionado

Pon en `MEJOR_GRUPOS` / `MEJOR_PARAMS` solo los cambios con veredicto **mejora** en el bloque 4. Varios cambios que ayudan por
separado no siempre suman: la celda de comprobación los mide juntos con 2 semillas frente a la base del bloque 4.

In [ ]:
MEJOR_GRUPOS = ["alerta", "contexto"]   # + ["avion_tipos"] si mejora en el bloque 4
MEJOR_PARAMS = {}                       # p. ej. dict(lambdarank_truncation_level=3) si mejora en el bloque 4

In [ ]:
# Comprobación (~1 min). Se puede saltar si MEJOR es igual a la base del bloque 4.
cv_comb, peor_comb = experimento("b5/comb", MEJOR_GRUPOS, MEJOR_PARAMS, nota="combinación de ganadores")
print(f"combinación: {cv_comb:.4f} (peor controlador {peor_comb:.3f})")

**Conclusión bloque 5 · (rellenar tras ejecutar)**

## 6 · Versión para el stacking

Entrena `MEJOR` con **3 semillas** y guarda en el formato común. Juanma lo usa en el stacking (**nadie más sube a Kaggle**).
~3 min. Sobrescribe los archivos de `lgb_adri`:
- `oof/lgb_adri.csv` — predicciones OOF (stacking de Juanma, análisis de errores)
- `preds/lgb_adri.csv` y `submissions/sub_lgb_adri.csv` — predicciones de test (media de las semillas)
- `models/lgb_adri_s<semilla>.txt` — **los pesos**, entrenados con todo train+validation
- `models/lgb_adri_config.json` — grupos, parámetros y features usados

In [ ]:
params = {**PARAMS, **MEJOR_PARAMS}
d, F = build(lab, MEJOR_GRUPOS)
oof, h, por_fold = cv(d, F, params, SEEDS_FINAL)
peor = h.groupby(ATCO.reindex(h.index)).mean().min()
print(f"CV top-3 final ({len(SEEDS_FINAL)} semillas): {h.mean():.4f}  peor controlador {peor:.3f}"
      f"  (folds: {' '.join(f'{v:.3f}' for v in por_fold)})")
pd.DataFrame([dict(hora=time.strftime("%H:%M"), nombre=f"FINAL {NAME}", grupos="+".join(MEJOR_GRUPOS) or "-",
                   params_cambiados=str(MEJOR_PARAMS), n_features=len(F), cv_top3=round(h.mean(), 4),
                   std_folds=round(por_fold.std(), 4),
                   nota=f"{len(SEEDS_FINAL)} semillas. peor_atco {peor:.3f}. Modelo guardado")]
             ).to_csv(LOG, mode="a", header=False, index=False)
d[["sample_id", "row_index"]].assign(score=oof).to_csv(ROOT / "oof" / f"{NAME}.csv", index=False)

dt, _ = build(test, MEJOR_GRUPOS)
score = np.zeros(len(dt))
for s in SEEDS_FINAL:
    m = fit(d, F, {**params, "random_state": s})
    m.booster_.save_model(str(ROOT / "models" / f"{NAME}_s{s}.txt"))
    score += m.predict(dt[F]) / len(SEEDS_FINAL)
json.dump(dict(groups=MEJOR_GRUPOS, params=params, seeds=SEEDS_FINAL, features=F, cv_top3=round(h.mean(), 4)),
          open(ROOT / "models" / f"{NAME}_config.json", "w"), indent=1)

pred = dt[["sample_id", "row_index"]].assign(score=score)
pred.to_csv(ROOT / "preds" / f"{NAME}.csv", index=False)
sub = (pred.sort_values(["sample_id", "score"], ascending=[True, False]).groupby("sample_id").row_index
           .apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction").reset_index())
sub.to_csv(ROOT / "submissions" / f"sub_{NAME}.csv", index=False)
print("Guardado:", NAME, "| grupos:", MEJOR_GRUPOS, "| params:", MEJOR_PARAMS or "base", "| test:", len(sub), "muestras")

In [ ]:
# Comprobación con el scorer oficial sobre las muestras de validation.csv: debe coincidir con nuestro hit3
val_ids = set(pd.read_csv(DATA / "validation.csv", usecols=["sample_id"]).sample_id)
o = d[["sample_id", "row_index"]].assign(score=oof)
o = o[o.sample_id.isin(val_ids)].sort_values(["sample_id", "score"], ascending=[True, False])
val_sub = o.groupby("sample_id").row_index.apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction")
val_sub.reset_index().to_csv(ROOT / "submissions" / f"val_{NAME}.csv", index=False)
print(f"hit3 propio en validation: {h[h.index.isin(val_ids)].mean():.4f}")
r = subprocess.run([sys.executable, DATA / "scorer.py", "--labelled", DATA / "validation.csv",
                    ROOT / "submissions" / f"val_{NAME}.csv"], capture_output=True, text=True)
print("scorer oficial:", r.stdout or r.stderr)

**Resultado de la versión para el stacking · (rellenar tras ejecutar)**

| | CV top-3 | peor controlador | validation.csv (scorer oficial) |
|---|---|---|---|
| base (columnas originales) | 0.657 | | |
| `lgb_adri` anterior (alerta + contexto, 2 semillas) | 0.657 | | 0.677 |
| **`lgb_adri` nuevo** | | | |

**Cómo usan los demás el modelo guardado (sin entrenar)** — necesitan las funciones de *Features* (`build`):
```python
cfg = json.load(open("models/lgb_adri_config.json"))
dt, F = build(test, cfg["groups"])
score = np.mean([lgb.Booster(model_file=f"models/lgb_adri_s{s}.txt").predict(dt[F]) for s in cfg["seeds"]], axis=0)
```

## 7 · Siguientes ideas (no implementadas todavía)

Por orden de prioridad:
- **Ranking por avión (dos etapas):** el 83 % de los fallos es elegir mal el avión. Un primer modelo que ordene **aviones**
  (una fila por avión, con lo que tiene abierto) y un segundo que elija la tarea dentro del avión. Está en `EQUIPO.md` como
  tarea de Juanma (`m_two_stage.py`): coordinar antes de hacerlo aquí.
- **Barrido de parámetros más amplio** (`num_leaves`, `learning_rate` × árboles, `min_child_samples`): en `EQUIPO.md` era tarea de Pau.
- **XGBoost / CatBoost** (`rank:pairwise`, `YetiRank`): no para ganar solos, sino para dar variedad al stacking.
- **Si se permiten otros snapshots:** recuperar `f_next_input` (commit `67a15bc`): con él el LightGBM llegaba a 0.83.

## Historial

In [ ]:
historial(30)